## Question 2: Handwritten Character Classification using Transfer Learning - SOLUTION

You are provided with the **EMNIST Dataset** (Extended MNIST), which contains handwritten characters. We will use the **"letters"** split which contains 26 letter classes (A-Z).

Your objective is to **fine-tune a pretrained EfficientNetV2-Small** model to classify handwritten letters.

Your work will be evaluated based on the completion of the following tasks:

# Part 1: Load and Prepare Data 

**Tasks:**

- Complete the transforms by adding:
  - Resize images to 28x28
  - Convert to Tensor
  - Normalize using ImageNet mean and std
- Create DataLoaders for training and testing
- Display some sample images with their labels

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Get a batch of training data
images, labels = next(iter(train_loader))

# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Show images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))
    # Denormalize for display
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f"Label: {letters[labels[i].item()-1]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part 2: Load Pretrained Model and Adapt

**Tasks:**

- Load pretrained EfficientNetV2-Small
- **Freeze the backbone** (feature extractor) - we only want to train the classifier head
- Replace the classifier head to match the number of classes in EMNIST letters (26 classes)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Load pretrained EfficientNetV2-Small
model = efficientnet_v2_s(pretrained=True)

# Freeze the backbone (all layers except classifier)
for param in model.features.parameters():
    param.requires_grad = False

# Replace the classifier head
# EfficientNetV2-S classifier: (dropout, Linear(1280, 1000))
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

print("Model classifier head replaced!")
print(f"New classifier: {model.classifier}")

# Verify only classifier is trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,}")

# Part 3: Training and Validation Functions 

**Tasks:**

- Define the training loop function
- Define the validation loop function

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        # EMNIST letters labels are 1-26, need to make them 0-25
        labels = labels - 1
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # Adjust labels
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Part 4: Training 

**Tasks:**

- Set up device, model, loss function, and optimizer
- Train the model
- Plot the training and validation losses
- Plot the training and validation Accuracy

In [ ]:
import torch.optim as optim

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  

num_epochs = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

In [ ]:
# Plot Loss and Accuracy
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Acc')
plt.plot(val_accuracies, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Part 5: Bonus - Test Time Augmentation (TTA) 

**Task:**

Implement Test Time Augmentation (TTA) to improve predictions!

**Steps to implement TTA:**

1. Copy the validation function you created above

2. Inside the function, for each batch, get predictions for 3 versions of the images:
   - Predictions from the **original** images
   - Predictions from **horizontally flipped** images (to get the images, use `h_flipped = torch.flip(images, dims=[3])`)
   - Predictions from **vertically flipped** images (to get the images, use `v_flipped = torch.flip(images, dims=[2])`)

3. **Average** the 3 predictions together, then pass the average to the criterion and get the accuracy

In [ ]:
# Step 1: Copy the validation function
def validate_with_tta(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # Adjust labels
            
            # Step 2: Get predictions for 3 versions of images
            # Original prediction
            outputs_original = model(images)
            
            # Horizontal flip prediction
            h_flipped = torch.flip(images, dims=[3])
            outputs_hflip = model(h_flipped)
            
            # Vertical flip prediction
            v_flipped = torch.flip(images, dims=[2])
            outputs_vflip = model(v_flipped)
            
            # Step 3: Average the 3 predictions, then pass to criterion
            outputs_avg = (outputs_original + outputs_hflip + outputs_vflip) / 3
            loss = criterion(outputs_avg, labels)
            total_loss += loss.item()
            
            _, predicted = outputs_avg.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Run TTA evaluation
tta_loss, tta_acc = validate_with_tta(model, test_loader, criterion, device)

print(f"Validation with TTA: Loss={tta_loss:.4f}, Accuracy={tta_acc:.2f}%")